# Chapter 18: Where to Go Next

*Data Analytics with Python and Jupyter Notebook*

Small, runnable demonstrations for the next steps after this book: making pandas use less memory, processing files in chunks, SQL with window functions, and turning analysis into a reusable function. The Polars, DuckDB and Streamlit examples from the chapter are not included here because those packages are not part of the book's standard installation; try them after `pip install polars duckdb streamlit`.

## 18.1 Look how far you have come

We will use the Q1 2026 running dataset for the small demonstrations, so the companion notebook runs on its own.

In [1]:
import sqlite3

import numpy as np
import pandas as pd

pd.set_option("display.width", 80)
sales = pd.read_csv("../data/bean_there_sales.csv", parse_dates=["date"])
sales.shape

(1620, 7)

## 18.2 When data gets big

### Level 1: make pandas use less memory

Before reaching for new tools, check how much memory your DataFrame actually uses. To simulate a bigger file, we stack 500 copies of the Q1 data, giving 810,000 rows:

In [2]:
big = pd.concat([sales] * 500, ignore_index=True)
mb = big.memory_usage(deep=True) / 1e6
print(mb.round(1))
print(f"Total: {mb.sum():.0f} MB for {len(big):,} rows")

Index          0.0
date           6.5
store         47.0
product       47.4
category      44.6
quantity       6.5
unit_price     6.5
revenue        6.5
dtype: float64
Total: 165 MB for 810,000 rows


Those text columns hold only a handful of distinct values: three stores, six products, two categories. The **category** dtype (Chapter 7) stores each distinct value once and replaces every row with a small integer code:

In [3]:
slim = big.astype({"store": "category", "product": "category",
                   "category": "category"})
slim_mb = slim.memory_usage(deep=True).sum() / 1e6
print(f"Before: {mb.sum():.0f} MB   After: {slim_mb:.0f} MB")

Before: 165 MB   After: 28 MB


One line cut memory use by more than 80%, and every pandas method you know still works. Two other easy savings: read only the columns you need with `usecols=`, and choose dtypes as you read, so the large version never exists in memory at all:

In [4]:
small = pd.read_csv("../data/bean_there_sales.csv",
                    usecols=["date", "store", "revenue"],
                    dtype={"store": "category"},
                    parse_dates=["date"])
small.dtypes

date       datetime64[us]
store            category
revenue           float64
dtype: object

### Level 2: process the file in pieces

If a file is too big to load even after slimming, you can read it in **chunks**. `pd.read_csv(..., chunksize=n)` returns an iterator that hands you `n` rows at a time. You compute a partial result from each chunk and combine the partial results at the end:

In [5]:
partials = []
for chunk in pd.read_csv("../data/bean_there_sales.csv", chunksize=500):
    partials.append(chunk.groupby("store")["revenue"].sum())

by_store = pd.concat(partials).groupby(level=0).sum()
print(len(partials), "chunks")
by_store

4 chunks


store
Downtown      70205.3
Riverside     40138.2
University    50570.4
Name: revenue, dtype: float64

### Level 3: tools built for bigger data

## 18.3 SQL: the other language of data

If you learn only one more thing after this book, make it SQL. Almost every organization keeps its data in a database, and SQL is how you ask a database questions. Chapter 6 showed how to read a SQL query into pandas. Here we go a little further, using Python's built-in `sqlite3` module and an in-memory database:

In [6]:
con = sqlite3.connect(":memory:")
sales.to_sql("sales", con, index=False)

query = """
SELECT store,
       ROUND(SUM(revenue), 2) AS revenue,
       SUM(quantity)          AS units
FROM sales
GROUP BY store
ORDER BY revenue DESC
"""
pd.read_sql_query(query, con)

,store,revenue,units
0,Downtown,70205.3,18013
1,University,50570.4,12979
2,Riverside,40138.2,10335


### Window functions and CTEs

Here is "rank the products within each store by units sold" in one query:

In [7]:
query = """
WITH totals AS (
    SELECT store, product, SUM(quantity) AS units
    FROM sales
    GROUP BY store, product
)
SELECT store, product, units,
       RANK() OVER (PARTITION BY store ORDER BY units DESC) AS rnk
FROM totals
ORDER BY store, rnk
"""
ranked = pd.read_sql_query(query, con)
ranked[ranked["rnk"] <= 2]

,store,product,units,rnk
0,Downtown,Latte,4673,1
1,Downtown,Cappuccino,3175,2
6,Riverside,Latte,2599,1
7,Riverside,Cappuccino,1801,2
12,University,Latte,3286,1
13,University,Cappuccino,2319,2


Another classic is comparing each month with the previous one using `LAG`, the SQL cousin of pandas' `shift()` (Chapter 9):

In [8]:
query = """
WITH monthly AS (
    SELECT strftime('%Y-%m', date) AS month,
           SUM(revenue) AS revenue
    FROM sales
    GROUP BY month
)
SELECT month,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1) AS change_pct
FROM monthly
"""
pd.read_sql_query(query, con)

,month,revenue,change_pct
0,2026-01,51411.20,NaN
1,2026-02,50254.75,-2.2
2,2026-03,59247.95,17.9


## 18.4 Automation: let the computer do it again

### From notebook to function

The first step is always the same: wrap the analysis in a **function** with clear inputs (Chapter 3). Here is a small monthly summary for the owner:

In [9]:
def summarize(sales, month):
    """Return the headline numbers for one month as a dictionary."""
    rows = sales[sales["date"].dt.strftime("%Y-%m") == month]
    by_store = rows.groupby("store")["revenue"].sum()
    by_product = rows.groupby("product")["quantity"].sum()
    return {
        "days": rows["date"].nunique(),
        "revenue": rows["revenue"].sum(),
        "best_store": by_store.idxmax(),
        "best_product": by_product.idxmax(),
    }


for month in ["2026-01", "2026-02", "2026-03"]:
    s = summarize(sales, month)
    print(month, s["days"], f"${s['revenue']:,.2f}", s["best_store"])

2026-01 31 $51,411.20 Downtown
2026-02 28 $50,254.75 Downtown
2026-03 31 $59,247.95 Downtown


## 18.11 Exercises

Open `ch18_where_to_go_next.ipynb`, which already loads `sales` and the in-memory SQL database `con`. Solutions are in Appendix D.

1. Load `bean_there_sales.csv` with `store`, `product` and `category` read directly as category dtype. How much memory does it use compared with the default load? *Hint:* pass `dtype=` to `pd.read_csv` and compare `memory_usage(deep=True).sum()`.
2. Use `chunksize=400` to compute the **total quantity of each product** without loading the file in one go. Check that your answer matches a normal `groupby`.
3. Write a SQL query that returns the **best single day** for revenue at each store. *Hint:* total revenue by store and date in a CTE, then use `RANK() OVER (PARTITION BY store ORDER BY ...)` and keep rank 1.
4. Write a SQL query for Cold Brew's **quantity per month** and the change from the previous month, using `LAG`. Compare with Chapter 1's Exercise 4.
5. Extend `summarize` with an optional `store=None` argument so it can report on one store. Use it to print March 2026 revenue for each store.
6. Add a check to `summarize` that raises a `ValueError` with a helpful message when the month has no data (for example `"2025-12"`). Test it with `try` and `except`.
7. **Think about it:** choose a public dataset on a topic you care about and write the four or five questions you would ask, Chapter 17 style, with the decision each one informs. (No code needed; the solution shows an example.)